# Customer Churn and Revenue Retention Analysis

## Context

A California telecom provider sells phone and internet service to residential customers. Leadership has seen customers leave and wants to know why, how much revenue is going with them, and whether anything the company already records could help it keep more of them.

## My role

I am the revenue analyst on the finance and revenue team. My focus is how much recurring revenue churn removes, what is driving it, and which of those drivers the company can actually influence.

## What leadership asked

1. What reasons did customers give for leaving, and which cost the most revenue?
2. Which of those reasons could the company have influenced, and which could it not?
3. At what point in the customer relationship does churn happen?
4. Which customer types, contracts, services and billing habits go with higher or lower churn?
5. How much does price matter?
6. What is missing from the records that would be needed to answer these questions properly?

## About the data

The analysis uses IBM's public Telco Customer Churn sample (7,043 customers, one row per customer). It is synthetic, so the results demonstrate method and are not findings about a real company.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter   # keeps percentage axes on 0-100%
from scipy.stats import chi2_contingency, mannwhitneyu   # compare rates; compare skewed charges
import statsmodels.formula.api as smf             # optional: controlled check for confounding

pd.set_option("display.max_columns", 80)

# Part 1. Data audit

The audit establishes what the data contains before any change is made. Every check runs against `customers_raw`, which is never modified, and the section ends with a summary of what needs fixing in Part 2.

### 1.1 Load and first look

The file is loaded once. `customers_raw` is the untouched original and `customers` is the working copy, so every later change can be compared against the source.

In [18]:
customers_raw = pd.read_excel("Telco_customer_churn.xlsx")


# Work on copies from here on - the _raw versions are never touched again (audit trail).
customers = customers_raw.copy()

print(customers.shape)
print(customers.dtypes)
customers.head()

(7043, 33)
CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


**Finding:** 7,043 rows and 33 columns. Total Charges is stored as text (`object`) although it holds dollar amounts, and Zip Code is stored as a number although it is an identifier. Both are addressed in Part 2.

### 1.2 Grain and keys

Every rate in this analysis is calculated per customer, so duplicates would inflate churn and any later merge could silently multiply rows.

In [20]:
print("CustomerID unique:", customers_raw["CustomerID"].is_unique)
print("Duplicate IDs:", customers_raw["CustomerID"].duplicated().sum())
print("Fully duplicated rows:", customers_raw.duplicated().sum())
print("Values in Count:", customers_raw["Count"].unique())

CustomerID unique: True
Duplicate IDs: 0
Fully duplicated rows: 0
Values in Count: [1]


**Finding:** CustomerID is unique, there are no duplicate rows, and the `Count` column is 1 for every row. The grain is one row per customer, so no deduplication is needed.

### 1.3 Missing values, real and disguised

`isna()` only finds true blanks. In files like this one, missing data often hides as an empty string or a placeholder, so I check for both.

In [21]:
na_counts = customers_raw.isna().sum()
print("Real NaN by column (only columns with any):")
print(na_counts[na_counts > 0])

def blank_like(s):
    if pd.api.types.is_numeric_dtype(s):
        return 0
    t = s.dropna().astype(str).str.strip()
    return (t == "").sum() + t.str.lower().isin(["na", "n/a", "null", "none", "unknown", "?", "-"]).sum()

blank_counts = pd.Series({c: blank_like(customers_raw[c]) for c in customers_raw.columns})
print("\nBlank or placeholder-like text by column:")
print(blank_counts[blank_counts > 0])

Real NaN by column (only columns with any):
Churn Reason    5174
dtype: int64

Blank or placeholder-like text by column:
Total Charges    11
dtype: int64


**Finding:** The only true missing values are in Churn Reason (5,174), which exactly matches the number of customers who did not churn, so it is blank by design and not an error. The only disguised missing values are in Total Charges (11 blank strings), which is why that column loaded as text.

### 1.4 The Total Charges problem

The 11 rows that could not be read as numbers are inspected to see which customers they are before deciding how to treat them.

In [22]:
tc_num = pd.to_numeric(customers_raw["Total Charges"], errors="coerce")
bad_tc = customers_raw.loc[tc_num.isna(), ["CustomerID", "Tenure Months", "Monthly Charges",
                                           "Total Charges", "Contract", "Churn Label"]]
print("Total Charges that cannot be read as a number:", len(bad_tc))
bad_tc

Total Charges that cannot be read as a number: 11


,CustomerID,Tenure Months,Monthly Charges,Total Charges,Contract,Churn Label
2234,4472-LVYGI,0,52.55,,Two year,No
2438,3115-CZMZD,0,20.25,,Two year,No
2568,5709-LVOEQ,0,80.85,,Two year,No
2667,4367-NUYAO,0,25.75,,Two year,No
2856,1371-DWPAZ,0,56.05,,Two year,No
4331,7644-OMVMY,0,19.85,,Two year,No
4687,3213-VVOLG,0,25.35,,Two year,No
5104,2520-SGTTA,0,20.00,,Two year,No
5719,2923-ARZLG,0,19.70,,One year,No
6772,4075-WKNIU,0,73.35,,Two year,No


**Finding:** All 11 customers have a tenure of 0 months, none has churned, and 10 are on two-year contracts. They are new customers who have not yet been billed, not data entry errors. Part 2 converts the column to numeric, flags these rows, and leaves them missing instead of filling in 0 or an average.

### 1.5 Categorical values

Every value in each categorical column is listed. This catches spelling variants and spacing problems, and exposes labels that look like missing data but are not.

In [23]:
cat_cols = [c for c in customers_raw.columns
            if customers_raw[c].nunique() <= 25 and c not in ["Count", "Churn Value", "Churn Score"]]
for c in cat_cols:
    print(f"\n{c}")
    print(customers_raw[c].value_counts(dropna=False).to_string())

variant_issues = []
for c in cat_cols:
    s = customers_raw[c].dropna().astype(str)
    if s.str.strip().str.lower().nunique() < s.nunique():
        variant_issues.append(c)
print("\nColumns where spacing or capitalization splits a label:", variant_issues)


Country
Country
United States    7043

State
State
California    7043

Gender
Gender
Male      3555
Female    3488

Senior Citizen
Senior Citizen
No     5901
Yes    1142

Partner
Partner
No     3641
Yes    3402

Dependents
Dependents
No     5416
Yes    1627

Phone Service
Phone Service
Yes    6361
No      682

Multiple Lines
Multiple Lines
No                  3390
Yes                 2971
No phone service     682

Internet Service
Internet Service
Fiber optic    3096
DSL            2421
No             1526

Online Security
Online Security
No                     3498
Yes                    2019
No internet service    1526

Online Backup
Online Backup
No                     3088
Yes                    2429
No internet service    1526

Device Protection
Device Protection
No                     3095
Yes                    2422
No internet service    1526

Tech Support
Tech Support
No                     3473
Yes                    2044
No internet service    1526

Streaming TV
Streaming T

**Finding:** No spelling or capitalization variants. Two labels need care. "No phone service" (682) matches Phone Service = No exactly, and "No internet service" (1,526) appears in all six add-on columns and matches Internet Service = No exactly. These are genuine answers, not missing data, and they show which customers could not have bought a service at all. Churn Reason has 20 distinct reasons, which will need grouping later.

### 1.6 Range checks

Impossible values are checked for: tenure outside a plausible range, non-positive charges, churn flags other than 0 or 1, and coordinates outside California.

In [24]:
print(customers_raw[["Tenure Months", "Monthly Charges"]].describe().round(2))
print("\nTenure outside 0-72:", (~customers_raw["Tenure Months"].between(0, 72)).sum())
print("Monthly Charges <= 0:", (customers_raw["Monthly Charges"] <= 0).sum())
print("Churn Value not 0/1:", (~customers_raw["Churn Value"].isin([0, 1])).sum())
print("Outside California bounding box:",
      (~customers_raw["Latitude"].between(32.5, 42.1) | ~customers_raw["Longitude"].between(-124.5, -114.1)).sum())

       Tenure Months  Monthly Charges
count        7043.00          7043.00
mean           32.37            64.76
std            24.56            30.09
min             0.00            18.25
25%             9.00            35.50
50%            29.00            70.35
75%            55.00            89.85
max            72.00           118.75

Tenure outside 0-72: 0
Monthly Charges <= 0: 0
Churn Value not 0/1: 0
Outside California bounding box: 0


**Finding:** None. Tenure runs from 0 to 72 months, monthly charges from $18.25 to $118.75, Churn Value is only 0 or 1, and every coordinate falls inside California.

### 1.7 Internal consistency

Columns that describe the same thing should agree. The pairs that should match are tested, and Total Charges is compared with what tenure times monthly charges would predict.

In [25]:
chk = {}
chk["Churn Label matches Churn Value"] = ((customers_raw["Churn Label"] == "Yes") == (customers_raw["Churn Value"] == 1)).all()
chk["Churn Reason filled only for churners"] = (customers_raw["Churn Reason"].notna() == (customers_raw["Churn Value"] == 1)).all()
chk["Multiple Lines agrees with Phone Service"] = ((customers_raw["Phone Service"] == "No") == (customers_raw["Multiple Lines"] == "No phone service")).all()
addons = ["Online Security", "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies"]
chk["Add-ons agree with Internet Service"] = all(
    ((customers_raw["Internet Service"] == "No") == (customers_raw[c] == "No internet service")).all() for c in addons)
chk["Lat Long matches Latitude/Longitude"] = (
    customers_raw["Lat Long"] == customers_raw["Latitude"].astype(str) + ", " + customers_raw["Longitude"].astype(str)).all()
print(pd.Series(chk))

ratio = tc_num / (customers_raw["Tenure Months"] * customers_raw["Monthly Charges"])
print("\nTotal Charges / (Tenure x Monthly Charges):")
print(ratio.describe().round(3))
print("Within 10% of tenure x monthly:", ratio.between(0.9, 1.1).mean().round(3))
print("Tenure 0 customers:", (customers_raw["Tenure Months"] == 0).sum(),
      "| of those with blank Total Charges:", (tc_num.isna() & (customers_raw["Tenure Months"] == 0)).sum())
print("Churners with tenure 0:", ((customers_raw["Tenure Months"] == 0) & (customers_raw["Churn Value"] == 1)).sum())

Churn Label matches Churn Value             True
Churn Reason filled only for churners       True
Multiple Lines agrees with Phone Service    True
Add-ons agree with Internet Service         True
Lat Long matches Latitude/Longitude         True
dtype: bool

Total Charges / (Tenure x Monthly Charges):
count    7032.000
mean        1.000
std         0.051
min         0.689
25%         0.980
50%         1.000
75%         1.020
max         1.573
dtype: float64
Within 10% of tenure x monthly: 0.944
Tenure 0 customers: 11 | of those with blank Total Charges: 11
Churners with tenure 0: 0


**Finding:** All five pairs agree. Total Charges is close to tenure times monthly charges for most customers (median ratio 1.00, 94.4% within 10%), with a range of 0.69 to 1.57. The remaining 6% likely reflect price changes during a customer's tenure, but nothing in the file confirms that. None of the 11 customers with tenure 0 has churned.

### 1.8 Columns that add no information

Columns with a single value, or that duplicate another column, add clutter without helping any question.

In [26]:
single = [c for c in customers_raw.columns if customers_raw[c].nunique() == 1]
print("Single-value columns:", single)
dup_cols = [(a, b) for i, a in enumerate(customers_raw.columns) for b in customers_raw.columns[i + 1:]
            if customers_raw[a].equals(customers_raw[b])]
print("Exact duplicate columns:", dup_cols)
print("Zip codes:", customers_raw["Zip Code"].nunique(),
      "| customers per zip (median):", customers_raw["Zip Code"].value_counts().median())

Single-value columns: ['Count', 'Country', 'State']
Exact duplicate columns: []
Zip codes: 1652 | customers per zip (median): 4.0


**Finding:** Count, Country and State each hold a single value (1, United States, California). No two columns are exact duplicates, although Lat Long repeats Latitude and Longitude combined. Zip Code has 1,652 distinct values with a median of 4 customers each, which is too sparse for zip-level analysis, so any geographic work will use city.

### 1.9 Extreme values

Extreme values are checked, and both ends of the tenure distribution are examined. A value is removed only if it is clearly wrong. An unusual but possible value is kept and flagged.

In [27]:
def iqr_flags(s):
    q1, q3 = s.quantile([0.25, 0.75]); i = q3 - q1
    return ((s < q1 - 3 * i) | (s > q3 + 3 * i)).sum()

for name, s in [("Tenure Months", customers_raw["Tenure Months"]),
                ("Monthly Charges", customers_raw["Monthly Charges"]),
                ("Total Charges", tc_num.dropna())]:
    print(f"{name}: beyond 3 x IQR = {iqr_flags(s)}")

print("\nTenure counts at the two ends:")
t = customers_raw["Tenure Months"].value_counts().sort_index()
print(t.head(3), t.tail(3))

Tenure Months: beyond 3 x IQR = 0
Monthly Charges: beyond 3 x IQR = 0
Total Charges: beyond 3 x IQR = 0

Tenure counts at the two ends:
Tenure Months
0     11
1    613
2    238
Name: count, dtype: int64 Tenure Months
70    119
71    170
72    362
Name: count, dtype: int64


**Finding:** No values fall outside 3 times the interquartile range in tenure, monthly charges, or total charges, so nothing is removed or flagged. Tenure piles up at both ends: 613 customers at 1 month, and 362 at 72 months compared with 119 at 70 and 170 at 71. The 72 looks like a ceiling on how far back the records go, so long-tenure customers are cut off at that point. Both patterns matter for the tenure question later.

### 1.10 Audit summary

All checks in one table, with the action each one needs.

In [28]:
audit = pd.DataFrame([
    ["Duplicate customers", int(customers_raw["CustomerID"].duplicated().sum()), "None needed"],
    ["Real NaN values", int(na_counts.sum()), "None needed; blank by design for the 5,174 customers who stayed"],
    ["Total Charges stored as text", len(bad_tc), "Convert to numeric; flag and leave the blanks missing"],
    ["Impossible values", 0, "None needed"],
    ["Inconsistent column pairs", int((~pd.Series(chk)).sum()), "None needed"],
    ["Single-value columns", len(single), "Drop from working table"],
    ["Extreme values (3 x IQR, Monthly Charges)", int(iqr_flags(customers_raw["Monthly Charges"])), "Keep, no removal needed"],
], columns=["Check", "Items affected", "Action"])
audit

,Check,Items affected,Action
0,Duplicate customers,0,None needed
1,Real NaN values,5174,"None needed; blank by design for the 5,174 cus..."
2,Total Charges stored as text,11,Convert to numeric; flag and leave the blanks ...
3,Impossible values,0,None needed
4,Inconsistent column pairs,0,None needed
5,Single-value columns,3,Drop from working table
6,"Extreme values (3 x IQR, Monthly Charges)",0,"Keep, no removal needed"


**Conclusion:** The data is structurally sound. There are no duplicates, no impossible values, and no contradictions between columns. Three things need handling in Part 2: Total Charges is stored as text with 11 blanks, three columns carry no information, and the 20 churn reasons will need grouping for analysis. Two limits carry forward: tenure is capped at 72 months, and Total Charges does not match tenure times monthly charges for about 6% of customers.

# Part 2. Cleaning

The audit found no duplicates, no impossible values, and no contradictions between columns. What it did find is a text column that should be numeric, three columns that carry no information, and two columns I do not want to use. Each fix below is made on `customers`, one step at a time, and every change is recorded in a log at the end.

### 2.1 Standardize column names

Column names are converted to lowercase with underscores. Consistent names reduce typing errors and make later code easier to read. No data values change.

In [29]:
customers.columns = (customers.columns
                     .str.strip()
                     .str.lower()
                     .str.replace(" ", "_"))
customers = customers.rename(columns={"customerid": "customer_id"})
print(list(customers.columns))

['customer_id', 'count', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_label', 'churn_value', 'churn_score', 'cltv', 'churn_reason']


**Result:** All 33 columns follow a single naming style (for example `tenure_months`, `monthly_charges`, `churn_value`). `CustomerID` is now `customer_id`.

### 2.2 Convert Total Charges to numeric

Total Charges loaded as text because 11 values are blank strings. Converting the column turns them into missing values, and a flag column marks them.

These 11 customers have a tenure of 0 months and have not yet been billed. The values stay missing: 0 would imply a customer billed nothing, and an average would invent a charge that does not exist. Dropping the rows would discard 11 customers with valid contract and service data.

In [30]:
customers["total_charges"] = pd.to_numeric(customers["total_charges"], errors="coerce")
customers["flag_total_charges_blank"] = customers["total_charges"].isna().astype(int)

print(customers["total_charges"].dtype, "| flagged:", customers["flag_total_charges_blank"].sum())
print("Tenure of flagged customers:", customers.loc[customers["flag_total_charges_blank"] == 1, "tenure_months"].unique())
customers["total_charges"].describe().round(2)

float64 | flagged: 11
Tenure of flagged customers: [0]


count    7032.00
mean     2283.30
std      2266.77
min        18.80
25%       401.45
50%      1397.48
75%      3794.74
max      8684.80
Name: total_charges, dtype: float64

**Result:** Total Charges is numeric. 11 customers are flagged, all with a tenure of 0 months. The remaining 7,032 values range from $18.80 to $8,684.80, with a median of $1,397.48.

### 2.3 Store Zip Code as text

A zip code is an identifier, not a quantity. Storing it as text prevents accidental arithmetic and preserves leading zeros that zip codes elsewhere can carry.

In [31]:
print("Digits per zip code:")
print(customers["zip_code"].astype(str).str.len().value_counts())

customers["zip_code"] = customers["zip_code"].astype(str).str.zfill(5)
print("\nNew type:", customers["zip_code"].dtype, "| examples:", customers["zip_code"].head(3).tolist())

Digits per zip code:
zip_code
5    7043
Name: count, dtype: int64

New type: str | examples: ['90003', '90005', '90006']


**Result:** All 7,043 zip codes already have 5 digits, so no values change. The conversion is a safeguard.

### 2.4 Drop columns with no information

Three columns hold a single value (`count`, `country`, `state`) and one repeats other columns (`lat_long` is `latitude` and `longitude` combined). They add width without informing any question. The originals remain in `customers_raw`.

In [32]:
drop_cols = ["count", "country", "state", "lat_long"]
before = customers.shape[1]
customers = customers.drop(columns=drop_cols)
print("Columns:", before, "->", customers.shape[1])

Columns: 34 -> 30


**Result:** Four columns removed. No rows lost.

### 2.5 Set aside model-generated columns

`churn_score` and `cltv` are outputs of IBM's models, not recorded customer behavior. `churn_score` separates churners from non-churners almost perfectly, so using it to explain churn would be circular. `cltv` is an estimate, not realized revenue.

Both columns move to a separate table, `model_outputs`, keyed by `customer_id`. They stay available for comparison but cannot enter the main analysis by accident.

In [33]:
model_outputs = customers[["customer_id", "churn_score", "cltv"]].copy()
customers = customers.drop(columns=["churn_score", "cltv"])
print("customers:", customers.shape, "| model_outputs:", model_outputs.shape)

customers: (7043, 28) | model_outputs: (7043, 3)


**Result:** The main table has 28 columns and 7,043 rows. `model_outputs` holds the two set-aside columns for the same 7,043 customers.

### 2.6 Keep "No phone service" and "No internet service" labels

These labels are genuine answers, not missing data. They show that a customer has no phone line or no internet and therefore could not buy the related services. Recoding them to "No" would merge customers who could not buy tech support with those who chose not to, and the tech support comparison depends on separating the two.

No change is made. Binary derived variables are added alongside the originals in Part 3.

In [34]:
print(customers["multiple_lines"].value_counts().to_dict())
print(customers["online_security"].value_counts().to_dict())

{'No': 3390, 'Yes': 2971, 'No phone service': 682}
{'No': 3498, 'Yes': 2019, 'No internet service': 1526}


**Result:** The original labels are intact: 682 customers have "No phone service" and 1,526 have "No internet service" in each add-on column.

### 2.7 Cleaning log

All decisions in one table, with the number of rows or columns affected and the flag column where one exists.

In [35]:
cleaning_log = pd.DataFrame([
    ["Column names", 33, "Lowercased, spaces to underscores; CustomerID renamed customer_id", "None"],
    ["Total Charges stored as text", int(customers["flag_total_charges_blank"].sum()), "Converted to numeric; blanks left missing", "flag_total_charges_blank"],
    ["Zip Code stored as a number", len(customers), "Converted to 5-character text", "None"],
    ["Single-value or duplicate columns", len(drop_cols), "Dropped: " + ", ".join(drop_cols), "None"],
    ["Model-generated columns (Churn Score, CLTV)", 2, "Moved to separate table model_outputs", "None"],
    ["Duplicate customers", 0, "None found", "None"],
    ["Impossible values or extreme values", 0, "None found", "None"],
], columns=["Issue", "Rows or columns affected", "Decision", "Flag column"])
cleaning_log

,Issue,Rows or columns affected,Decision,Flag column
0,Column names,33,"Lowercased, spaces to underscores; CustomerID ...",None
1,Total Charges stored as text,11,Converted to numeric; blanks left missing,flag_total_charges_blank
2,Zip Code stored as a number,7043,Converted to 5-character text,None
3,Single-value or duplicate columns,4,"Dropped: count, country, state, lat_long",None
4,"Model-generated columns (Churn Score, CLTV)",2,Moved to separate table model_outputs,None
5,Duplicate customers,0,None found,None
6,Impossible values or extreme values,0,None found,None


### 2.8 Validate the cleaned table

The cleaned table is checked against known values: customer count, unique IDs, exactly 11 missing Total Charges values, full churn counts, and none of the removed columns. A failed check stops the notebook before an error reaches the analysis.

In [36]:
assert customers.shape[0] == 7043
assert customers["customer_id"].is_unique
assert customers["total_charges"].isna().sum() == 11
assert customers["churn_value"].sum() == 1869
assert customers["churn_reason"].notna().sum() == 1869
assert not {"count", "country", "state", "lat_long", "churn_score", "cltv"} & set(customers.columns)
assert customers["zip_code"].str.len().eq(5).all()
assert set(model_outputs["customer_id"]) == set(customers["customer_id"])
print("All checks passed.", customers.shape)
customers.dtypes

All checks passed. (7043, 28)


customer_id                     str
city                            str
zip_code                        str
latitude                    float64
longitude                   float64
gender                          str
senior_citizen                  str
partner                         str
dependents                      str
tenure_months                 int64
phone_service                   str
multiple_lines                  str
internet_service                str
online_security                 str
online_backup                   str
device_protection               str
tech_support                    str
streaming_tv                    str
streaming_movies                str
contract                        str
paperless_billing               str
payment_method                  str
monthly_charges             float64
total_charges               float64
churn_label                     str
churn_value                   int64
churn_reason                    str
flag_total_charges_blank    

**Result:** All checks passed. The cleaned table has 7,043 customers and 28 columns: the remaining original columns plus one flag. Total Charges is numeric, Zip Code is text, and each of the 1,869 churned customers has a churn reason.

### 2.9 Save the cleaned files

Both tables are saved so later work starts from a fixed file instead of repeating the cleaning steps.

In [37]:
customers.to_csv("telco_customers_clean.csv", index=False)
model_outputs.to_csv("telco_model_outputs_set_aside.csv", index=False)
print(pd.read_csv("telco_customers_clean.csv", dtype={"zip_code": str}).shape)

(7043, 28)


**Conclusion:** Cleaning was light, consistent with the audit. Total Charges is numeric with 11 flagged blanks, four uninformative columns are removed, two model-generated columns are set aside, and no customer is dropped. 

# Part 3. Derived variables

Every variable the questions need is built in this one section. Each is defined once and checked before use, so every later question applies the same definition. All derived variables are added beside the original columns, never in place of them.

### 3.1 Tenure bands

Tenure in months is grouped into four bands: 0 to 12, 13 to 24, 25 to 48 and 49 to 72. Bands make churn timing easy to compare, and the first band isolates the first year, the period most telecom churn research focuses on. The 11 customers with a tenure of 0 fall in the first band.

In [38]:
customers["tenure_band"] = pd.cut(customers["tenure_months"],
                                  bins=[-1, 12, 24, 48, 72],
                                  labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"])

print(customers["tenure_band"].value_counts().sort_index())
print("Total:", customers["tenure_band"].value_counts().sum(), "| missing:", customers["tenure_band"].isna().sum())

tenure_band
0-12 months     2186
13-24 months    1024
25-48 months    1594
49-72 months    2239
Name: count, dtype: int64
Total: 7043 | missing: 0


**Result:** All 7,043 customers fall into a band and none is missing. The bands hold 2,186, 1,024, 1,594 and 2,239 customers. The first and last are the largest, matching the pile-up at both ends of the tenure distribution seen in the audit.

### 3.2 Service variables

Four variables describe what each customer buys:

- `has_phone` and `has_internet` are 1/0 flags for the two core services.
- `n_addons` counts how many of the six add-ons (security, backup, device protection, tech support, streaming TV, streaming movies) the customer has, from 0 to 6.
- `has_tech_support` is 1 or 0 for internet customers and missing for customers without internet. Customers without internet cannot buy tech support, so counting them as "no" would dilute the comparison with customers who could have bought it and did not.

In [39]:
addon_cols = ["online_security", "online_backup", "device_protection",
              "tech_support", "streaming_tv", "streaming_movies"]

customers["has_phone"] = (customers["phone_service"] == "Yes").astype(int)
customers["has_internet"] = (customers["internet_service"] != "No").astype(int)
customers["n_addons"] = (customers[addon_cols] == "Yes").sum(axis=1)
customers["has_tech_support"] = customers["tech_support"].map({"Yes": 1, "No": 0}).astype("Int64")

print(customers[["has_phone", "has_internet"]].sum().to_dict())
print("Add-ons per customer:", customers["n_addons"].value_counts().sort_index().to_dict())
print("Tech support:", customers["has_tech_support"].value_counts(dropna=False).to_dict())
print("Customers with add-ons but no internet:", ((customers["has_internet"] == 0) & (customers["n_addons"] > 0)).sum())

{'has_phone': 6361, 'has_internet': 5517}
Add-ons per customer: {0: 2219, 1: 966, 2: 1033, 3: 1118, 4: 852, 5: 571, 6: 284}
Tech support: {np.int64(0): 3473, np.int64(1): 2044, <NA>: 1526}
Customers with add-ons but no internet: 0


**Result:** 6,361 customers have phone service and 5,517 have internet. 2,219 customers have no add-ons: the 1,526 without internet plus 693 internet customers who bought none. At the other end, 284 have all six. `has_tech_support` splits into 2,044 yes, 3,473 no and 1,526 missing, which are the customers without internet. No customer without internet has an add-on, so the variables are consistent.

### 3.3 Automatic payment

The four payment methods fall into two behaviors: automatic (bank transfer and credit card) and manual (electronic check and mailed check). `is_autopay` captures that split, which is often treated as a sign of how settled a customer is. The original four-way `payment_method` is kept for the direct comparison.

In [40]:
customers["is_autopay"] = customers["payment_method"].isin(
    ["Bank transfer (automatic)", "Credit card (automatic)"]).astype(int)

pd.crosstab(customers["payment_method"], customers["is_autopay"])

is_autopay,0,1
payment_method,,
Bank transfer (automatic),0,1544
Credit card (automatic),0,1522
Electronic check,2365,0
Mailed check,1612,0


**Result:** 3,066 customers pay automatically and 3,977 pay manually. The cross-tab confirms the grouping is clean: both automatic methods map to 1 and both manual methods map to 0, with no overlap.

### 3.4 Household

`is_family` is 1 when the customer has a partner or dependents. Households with other people on the account may behave differently from single-person accounts, and one combined flag is simpler to compare than two overlapping columns. The separate partner and dependents columns remain available.

In [41]:
customers["is_family"] = ((customers["partner"] == "Yes") | (customers["dependents"] == "Yes")).astype(int)
customers["is_family"].value_counts().to_dict()

{1: 3704, 0: 3339}

**Result:** 3,704 customers have a partner or dependents and 3,339 have neither.

### 3.5 Annual revenue estimate

`annual_revenue_estimate` is monthly charges multiplied by 12. It converts a customer's monthly bill into a yearly revenue figure for sizing what a churned customer represents. It is an estimate, not recorded revenue: the data has no billing history, and it assumes the current monthly charge continues for a full year.

In [42]:
customers["annual_revenue_estimate"] = customers["monthly_charges"] * 12
customers["annual_revenue_estimate"].describe().round(2)

count    7043.00
mean      777.14
std       361.08
min       219.00
25%       426.00
50%       844.20
75%      1078.20
max      1425.00
Name: annual_revenue_estimate, dtype: float64

**Result:** The estimate ranges from $219 to $1,425 per customer per year, with a median of $844.20.

### 3.6 Churn reason groups

The 20 churn reasons are too fragmented to analyze directly, and several are close variants of each other. They are mapped to six groups through a lookup table, then merged onto the customers.

| Group | Reasons included |
|---|---|
| Competitor offer | Higher download speeds, more data, better offer, better devices |
| Support quality | Attitude of support person, attitude of service provider, poor expertise of phone support, poor expertise of online support |
| Price and charges | Price too high, extra data charges, long distance charges, lack of affordable download/upload speed |
| Product, service and network | Network reliability, product dissatisfaction, service dissatisfaction, limited range of services, lack of self-service on website |
| Life event | Moved, deceased |
| Unknown | Don't know |

`is_controllable` marks whether the company could have influenced the outcome: 1 for the first four groups, 0 for life events, and missing for unknown, since there is no basis to judge.

Three assignments are judgment calls: "lack of affordable download/upload speed" is grouped with price because the issue is affordability, and "service dissatisfaction" and "lack of self-service on website" are grouped with product, service and network because neither points to a specific cause. The classification is an assumption and should be treated as one in the findings.

In [43]:
reason_map = pd.DataFrame([
    ["Competitor offered higher download speeds", "Competitor offer", 1],
    ["Competitor offered more data", "Competitor offer", 1],
    ["Competitor made better offer", "Competitor offer", 1],
    ["Competitor had better devices", "Competitor offer", 1],
    ["Attitude of support person", "Support quality", 1],
    ["Attitude of service provider", "Support quality", 1],
    ["Poor expertise of phone support", "Support quality", 1],
    ["Poor expertise of online support", "Support quality", 1],
    ["Price too high", "Price and charges", 1],
    ["Extra data charges", "Price and charges", 1],
    ["Long distance charges", "Price and charges", 1],
    ["Lack of affordable download/upload speed", "Price and charges", 1],
    ["Network reliability", "Product, service and network", 1],
    ["Product dissatisfaction", "Product, service and network", 1],
    ["Service dissatisfaction", "Product, service and network", 1],
    ["Limited range of services", "Product, service and network", 1],
    ["Lack of self-service on Website", "Product, service and network", 1],
    ["Moved", "Life event", 0],
    ["Deceased", "Life event", 0],
    ["Don't know", "Unknown", pd.NA],
], columns=["churn_reason", "reason_group", "is_controllable"])
reason_map["is_controllable"] = reason_map["is_controllable"].astype("Int64")

actual = set(customers["churn_reason"].dropna().unique())
print("Reasons in data but not in lookup:", actual - set(reason_map["churn_reason"]))
print("Reasons in lookup but not in data:", set(reason_map["churn_reason"]) - actual)

rows_before = len(customers)
customers = customers.merge(reason_map, on="churn_reason", how="left", validate="many_to_one")
print("Rows before and after merge:", rows_before, len(customers))
print("\nReasons mapped per group:")
print(reason_map.groupby("reason_group").size())

Reasons in data but not in lookup: set()
Reasons in lookup but not in data: set()
Rows before and after merge: 7043 7043

Reasons mapped per group:
reason_group
Competitor offer                4
Life event                      2
Price and charges               4
Product, service and network    5
Support quality                 4
Unknown                         1
dtype: int64


**Result:** The lookup and the data match exactly: no reason is unmapped and none is unused. The merge keeps the row count at 7,043, so no customers were duplicated or lost. The 20 reasons map to six groups of 4, 4, 4, 5, 2 and 1 reasons.

### 3.7 Validate the derived variables

Silent errors are the main risk in derived variables, since the code runs without complaint even when the logic is wrong. The checks below confirm that the bands cover every customer, that missing values appear only where intended, and that the merge added no rows.

In [44]:
assert customers["customer_id"].is_unique and len(customers) == 7043
assert customers["tenure_band"].notna().all()
assert customers["tenure_band"].value_counts().sum() == 7043
assert customers["n_addons"].between(0, 6).all()
assert customers["has_tech_support"].isna().sum() == (customers["has_internet"] == 0).sum() == 1526
assert customers["reason_group"].notna().sum() == customers["churn_value"].sum() == 1869
assert customers.loc[customers["churn_value"] == 0, "reason_group"].isna().all()
print("All checks passed.", customers.shape)

All checks passed. (7043, 38)


**Result:** All checks passed. Every customer has a tenure band, `has_tech_support` is missing for exactly the 1,526 customers without internet, and `reason_group` is filled for exactly the 1,869 churned customers and no one else. The table now has 38 columns.

### 3.8 Derived variable dictionary

A reference table of every variable added in this section.

In [45]:
derived_dictionary = pd.DataFrame([
    ["tenure_band", "Months with the company, grouped: 0-12, 13-24, 25-48, 49-72", "tenure_months"],
    ["has_phone", "1 if the customer has phone service", "phone_service"],
    ["has_internet", "1 if the customer has DSL or fiber internet", "internet_service"],
    ["n_addons", "Number of the six add-ons the customer has (0 to 6)", "six add-on columns"],
    ["has_tech_support", "1 or 0 for internet customers; missing for customers without internet", "tech_support"],
    ["is_autopay", "1 if paid by bank transfer or credit card automatically", "payment_method"],
    ["is_family", "1 if the customer has a partner or dependents", "partner, dependents"],
    ["annual_revenue_estimate", "Monthly charges x 12; an estimate, not recorded revenue", "monthly_charges"],
    ["reason_group", "Churn reason grouped into six categories; churned customers only", "churn_reason"],
    ["is_controllable", "1 if the company could have influenced the reason, 0 if not, missing if unknown", "reason_group"],
], columns=["Variable", "Definition", "Built from"])
derived_dictionary

,Variable,Definition,Built from
0,tenure_band,"Months with the company, grouped: 0-12, 13-24,...",tenure_months
1,has_phone,1 if the customer has phone service,phone_service
2,has_internet,1 if the customer has DSL or fiber internet,internet_service
3,n_addons,Number of the six add-ons the customer has (0 ...,six add-on columns
4,has_tech_support,1 or 0 for internet customers; missing for cus...,tech_support
5,is_autopay,1 if paid by bank transfer or credit card auto...,payment_method
6,is_family,1 if the customer has a partner or dependents,"partner, dependents"
7,annual_revenue_estimate,"Monthly charges x 12; an estimate, not recorde...",monthly_charges
8,reason_group,Churn reason grouped into six categories; chur...,churn_reason
9,is_controllable,1 if the company could have influenced the rea...,reason_group


### 3.9 Save the analysis table

The table with all derived variables is saved so the analysis sections start from one fixed file.

In [46]:
customers.to_csv("telco_customers_analysis.csv", index=False)
print(customers.shape)

(7043, 38)


**Conclusion of Part 3:** Ten derived variables are in place and validated. The analysis table has 7,043 customers and 38 columns, ready for the questions. The next section establishes the baseline: overall churn, customer count and revenue, which every later comparison refers back to.

# Part 4. Baseline

Every later comparison needs a reference point. This section sets the overall figures: how many customers churned, what share of customers and of revenue that represents, and the estimated annual revenue involved. These values are stored in variables and reused in the questions that follow.

### 4.1 Customer churn

Churn rate is the share of customers who left. It is the benchmark against which every segment rate in the analysis is judged: a segment is only high or low relative to this figure.

In [47]:
n_customers = len(customers)
n_churned = int(customers["churn_value"].sum())
churn_rate = n_churned / n_customers

print(f"Customers: {n_customers:,}")
print(f"Churned: {n_churned:,} | Retained: {n_customers - n_churned:,}")
print(f"Customer churn rate: {churn_rate:.1%}")

Customers: 7,043
Churned: 1,869 | Retained: 5,174
Customer churn rate: 26.5%


**Result:** 1,869 of 7,043 customers churned, a churn rate of 26.5%. This is the benchmark for every segment comparison.

### 4.2 Revenue churn

Revenue churn measures how much recurring revenue left with those customers: the monthly charges of churned customers as a share of monthly charges across all customers. It is calculated from `monthly_charges`, since the data has no billing history. The annual figure multiplies the monthly amount by 12 and assumes each churned customer's current monthly charge would have continued for a full year, so it is an estimate and not recorded revenue.

In [48]:
mc_all = customers["monthly_charges"].sum()
mc_churned = customers.loc[customers["churn_value"] == 1, "monthly_charges"].sum()
revenue_churn_rate = mc_churned / mc_all

print(f"Monthly charges, all customers: ${mc_all:,.0f}")
print(f"Monthly charges, churned customers: ${mc_churned:,.0f}")
print(f"Revenue churn rate: {revenue_churn_rate:.1%}")
print(f"Annual revenue estimate, churned customers: ${mc_churned * 12:,.0f}")

print("\nAverage monthly charge:")
print(customers.groupby("churn_label")["monthly_charges"].mean().round(2))

Monthly charges, all customers: $456,117
Monthly charges, churned customers: $139,131
Revenue churn rate: 30.5%
Annual revenue estimate, churned customers: $1,669,570

Average monthly charge:
churn_label
No     61.27
Yes    74.44
Name: monthly_charges, dtype: float64


**Result:** Churned customers account for $139,131 of $456,117 in monthly charges, a revenue churn rate of 30.5%, or about $1.67 million in estimated annual revenue. Revenue churn (30.5%) is higher than customer churn (26.5%), which means churned customers paid more than average: $74.44 a month against $61.27 for retained customers. Part of that gap may come from the mix of services, since fiber internet costs more, so the price question compares customers within the same service type.

### 4.3 Baseline summary

The reference figures in one table.

In [49]:
baseline = pd.DataFrame([
    ["Customers in the file", f"{n_customers:,}"],
    ["Customers who churned", f"{n_churned:,}"],
    ["Customers retained", f"{n_customers - n_churned:,}"],
    ["Customer churn rate", f"{churn_rate:.1%}"],
    ["Monthly charges, all customers", f"${mc_all:,.0f}"],
    ["Monthly charges, churned customers", f"${mc_churned:,.0f}"],
    ["Revenue churn rate (share of monthly charges)", f"{revenue_churn_rate:.1%}"],
    ["Annual revenue estimate, churned customers", f"${mc_churned * 12:,.0f}"],
], columns=["Metric", "Value"])
baseline

,Metric,Value
0,Customers in the file,"7,043"
1,Customers who churned,"1,869"
2,Customers retained,"5,174"
3,Customer churn rate,26.5%
4,"Monthly charges, all customers","$456,117"
5,"Monthly charges, churned customers","$139,131"
6,Revenue churn rate (share of monthly charges),30.5%
7,"Annual revenue estimate, churned customers","$1,669,570"


**Conclusion of Part 4:** More than a quarter of customers (26.5%) and about 30% of monthly recurring revenue are gone, an estimated $1.67 million a year. The questions that follow explain why, starting with the reasons customers gave for leaving.